# The Vendor Fraud Review Op as an executable specification

The whitepaper's running example — the **Vendor Fraud Review Op** and its
Validation Strategy manifest (*The Distributed AI Economy*, Revision 9,
§5.6) — implemented on an RDF + SysML substrate with SKOS, PROV-O, EARL,
SHACL, and OpenSysML.

**Everything below is executed.** Every claim on this page is the output of
the code cell above it, produced live against the pinned source and the
committed files. Nothing is asserted by prose alone.

## 1 · Toolchain

The model runs on a pinned OpenSysML binary, fetched and verified against the
release's SHA256SUMS.

In [ ]:
import subprocess, pathlib
ROOT = pathlib.Path.cwd()
print(subprocess.run(["bash", "toolchain/get-sysml.sh"], capture_output=True, text=True).stdout or "already installed")
print(subprocess.run(["toolchain/bin/sysml", "--version"], capture_output=True, text=True).stdout.strip())

## 2 · The pinned source

Every definition and every gate below traces to one content-addressed snapshot
of the whitepaper. The hash the RDF declares must be the hash the file has.

In [ ]:
import hashlib, rdflib
pdf = next((ROOT / "sources").glob("*.pdf"))
actual = hashlib.sha256(pdf.read_bytes()).hexdigest()
g = rdflib.Graph()
g.parse("sources/sources.ttl"); g.parse("vocabulary/op-concepts.ttl")
VFR = "https://example.org/vfr#"
declared = str(next(g.objects(None, rdflib.URIRef(VFR + "contentHash"))))
print(f"snapshot file : {pdf.name}")
print(f"sha256(file)  : {actual}")
print(f"declared hash : {declared}")
print(f"match         : {actual == declared}")

## 3 · Vocabulary — his definitions, verbatim

Ten concepts, each with a `skos:definition` lifted exactly from the paper and
a section locator. The cell checks each definition against the PDF text
itself: an invented or paraphrased definition would print False.

In [ ]:
import re
from rdflib.namespace import SKOS
norm = lambda t: re.sub(r"\s+", "", re.sub(r"\([^)]*\)", "", t)).lower()
pdf_text = norm(subprocess.run(["pdftotext", "-layout", str(pdf), "-"], capture_output=True, text=True).stdout)
rows = []
for c in g.subjects(SKOS.definition, None):
    label = str(next(g.objects(c, SKOS.prefLabel)))
    definition = str(next(g.objects(c, SKOS.definition)))
    locator = str(next(g.objects(c, rdflib.URIRef(VFR + "locator"))))
    rows.append((locator, label, norm(definition) in pdf_text, definition))
for locator, label, verbatim, definition in sorted(rows):
    print(f"{locator:6} {label:24} verbatim: {verbatim}")
    print(f"       {definition[:96]}...")

## 4 · The manifest's gates as constraints

The §5.6 gates block, verbatim:

```yaml
gates:
  - if: confidence < 0.80
    then: human_review_required
  - if: consensus_disagreement > 0.25
    then: expert_review_required
  - if: sensitive_data_detected
    then: stop_and_escalate
  - if: vendor_risk == high
    then: human_approval_required
```

Each `if/then` line becomes a requirement with an `implies` constraint — the
direct material rendering. Two adjudicated interpretations are layered on top
(logged in `open-questions/`): `vendor_risk` and the confidence level are
enumerations `{high, low, unknown}` (GAP-01, GAP-03), and a then-clause names
an **obligation** — a required action, not yet materialized — which a concrete
action must discharge (GAP-02, the DISCHARGE-01 requirement). Two concrete
runs are bound to all requirements: a clean run (nothing fires) and the
escalated run (confidence 0.71 and a high vendor risk fire two gates, and
both obligations are discharged). The numerical parameters of the policy are
factored out as single-point definitions the gates reference — the defaults
are the manifest's literals, and the Track records the values in force per
run (his §5.3: "configuration parameters"). The satisfy sweep evaluates
everything.

In [ ]:
model = "model/vendor-fraud-review.sysml"
for gate_block in re.findall(r"requirement def <'GATE-\d+'>.*?\n    }", pathlib.Path(model).read_text(), re.S)[:1]:
    print(gate_block, "\n...")
r = subprocess.run(["toolchain/bin/sysml", model, "-validate", "-strict"], capture_output=True, text=True)
print(f"validate -strict exit code: {r.returncode}")
r = subprocess.run(["toolchain/bin/sysml", model, "-satisfy=RunConfigurations"], capture_output=True, text=True)
print(r.stdout.strip())
print(f"satisfy exit code: {r.returncode}")

The specification can also say **no**, in both ways a run can be wrong:
the gate fired but no obligation was raised, and the obligation was raised
but never discharged. The same algebra fails both, naming each violated
implication:

In [ ]:
r = subprocess.run(["toolchain/bin/sysml", "counterexamples/run-unattended.sysml", "-satisfy=RunConfigurations"], capture_output=True, text=True)
print(r.stdout.strip())
print(f"satisfy exit code: {r.returncode}")

## 5 · Conversion to RDF

The model converts to Turtle deterministically, and the gates survive the
conversion — so the same specification is queryable alongside the vocabulary
and the Track.

In [ ]:
c1 = subprocess.run(["toolchain/bin/sysml", model, "-convert", "ttl"], capture_output=True, text=True)
c2 = subprocess.run(["toolchain/bin/sysml", model, "-convert", "ttl"], capture_output=True, text=True)
mg = rdflib.Graph(); mg.parse(data=c1.stdout, format="turtle")
print(f"converted triples : {len(mg)}")
print(f"byte-stable       : {c1.stdout == c2.stdout}")
print(f"gates present     : {[gid for gid in ('GATE-01','GATE-02','GATE-03','GATE-04') if gid in c1.stdout]}")

## 6 · The Track

> "A Track is the durable record of an Op or Cog execution." (§5.3)

`track/run-001.trig` records the escalated run: three automatic Guard results
(passed, failed, and one honest *cantTell*), two fired Gates raising two
obligations, and one `earl:manual` approval by a named person that discharges
both obligations and generates the post-review case state (an action that
discharges an obligation must also mutate state). SHACL shapes derived from
the manifest's own `track.include` list check it — and refuse the
counterexample, where the obligations were raised and never discharged.

In [ ]:
from pyshacl import validate
def check(path):
    ds = rdflib.Dataset(default_union=True); ds.parse(path, format="trig")
    conforms, _, report = validate(ds, shacl_graph="shapes/track.shapes.ttl")
    return conforms, report
conforms, _ = check("track/run-001.trig")
print(f"run-001               conforms: {conforms}")
conforms, report = check("counterexamples/track-missing-approval.trig")
print(f"missing-approval      conforms: {conforms}")
print("\n".join(line for line in report.splitlines() if "Message" in line))

## 7 · Answering his own questions

§5.3 names the purposes a Track serves. Each is a query over the run graph,
not a reading exercise.

**Auditability** — "an organization can reconstruct why a decision was made":

In [ ]:
ds = rdflib.Dataset(default_union=True); ds.parse("track/run-001.trig", format="trig")
def show(rq):
    res = ds.query(pathlib.Path(rq).read_text())
    header = [str(v) for v in res.vars]
    print(" | ".join(header))
    for row in res:
        print(" | ".join(str(v) for v in row))
    return res
show("queries/auditability.rq");

**Governance** — "compliance teams can verify that required procedures were
followed". Rows are violations; empty means every obligation raised by a
fired Gate was discharged by a named human action. The same query catches
the counterexample:

In [ ]:
res = show("queries/governance.rq")
print(f"violations on run-001: {len(res)}")
cx = rdflib.Dataset(default_union=True); cx.parse("counterexamples/track-missing-approval.trig", format="trig")
cx_res = cx.query(pathlib.Path("queries/governance.rq").read_text())
print(f"violations on the counterexample: {len(cx_res)}")
for row in cx_res:
    print("  " + " | ".join(str(v) for v in row))

**Trust** — "users and customers can see that AI work was not merely
generated, but validated". The full outcome distribution, automatic and
manual, with *cantTell* visible rather than absorbed:

In [ ]:
show("queries/trust.rq");

**Interface** — where every value came from. The policy applies to
oracle-provided values; it is not their provider (GAP-05 ruling). For each
variable a gate evaluated: what service was called, what payload was sent,
what response code came back, and the response that carried the value. The
numerical precision lives inside the oracles or in documented threshold
rules; a reading nobody can source fails the shapes.

In [ ]:
res = ds.query(pathlib.Path("queries/interface.rq").read_text())
for row in res:
    d = row.asdict()
    print(f"{d['variable']} = {d['value']}")
    print(f"  service      : {d['service']}")
    print(f"  sent payload : {d['requestPayload']}")
    print(f"  responseCode : {d['responseCode']}")
    print(f"  response     : {d['responsePayload']}")

## 8 · Open questions

Where a literal parsing of the paper under-specifies what an executable
substrate requires, nothing was silently repaired: the choice in force is
marked provisional and logged. These entries are part of the demonstration.

In [ ]:
text = pathlib.Path("open-questions/computability-gaps.md").read_text()
for m in re.finditer(r"^## (GAP-\d+) — (.+?)$.*?\*\*Status:\*\* (\w+)", text, re.S | re.M):
    print(f"{m.group(1)}  [{m.group(3)}]  {m.group(2)}")

## 9 · Checks

One script runs everything on this page plus the test suite, and writes a
JSON report. The latest report:

In [ ]:
import json
report = pathlib.Path("checks/out/report.json")
if report.exists():
    print(json.dumps(json.loads(report.read_text()), indent=2))
else:
    print("no report yet — run: checks/run-checks.sh")